# Experimento 06: Validação de Sensibilidade ao Ruído (0,5) — Pan para Mathys (Cross-Dataset)

## Objetivo
Avaliar a capacidade de generalização e tolerância ao ruído do classificador celular **MLP treinado na referência limpa Pan**, ao realizar inferência sobre o dataset **Mathys**, cujos genes não medidos no Mathys recebem o valor sentinela `0,5` no espaço de colunas do Pan.

- **Dataset de Treino (Referência Limpa):** Pan (`57.457` células válidas, classes 1–7)
- **Dataset de Inferência Alvo:** Mathys (`45.572` células válidas, classes 1–7) alinhado ao Pan
- **Espaço de Projeção:** `61.541` genes $\to$ 600D com **`orthbase_pan_600d.npy`** (REUSO ESTRITO DA BASE DA REFERÊNCIA)
- **Ruído Injetado:** Genes do Pan ausentes no Mathys recebem `0,5`


In [1]:
import gc
import json
import os
import time
from pathlib import Path

import anndata as ad
import cupy as cp
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sweep
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential, load_model

# Garantir que o diretório de execução seja a raiz do projeto
if Path.cwd().name == "experimentos":
    os.chdir(Path.cwd().parent)
ROOT_DIR = Path.cwd()
DADOS_DIR = ROOT_DIR / "dados"

SEED = 42
DIM_PROJECAO = 600
CLASSES_VALIDAS = [1, 2, 3, 4, 5, 6, 7]
NOMES_CLASSES = [f"Classe {i}" for i in CLASSES_VALIDAS]

np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(1)

assert sweep.is_gpu_available(), "ERRO: GPU CUDA não detectada pelo pacote sweep!"
print(f"Diretório raiz ativo: {ROOT_DIR}")
print(f"Dispositivo GPU     : {sweep.get_gpu_device_name()}")
print(f"Versão SWeeP        : {getattr(sweep, 'DEFAULT_VERSION', 'v2.2.0')}")


2026-10-24 13:27:27.600009: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Diretório raiz ativo: /home/rasert/workspace/Teste_MlP_fujita_para_fujitafaltante_fujitapos_imputacao
Dispositivo GPU     : NVIDIA GeForce RTX 3060
Versão SWeeP        : SWeeP v2.9


In [2]:
def carregar_ensembl_ids(path_tsv: Path) -> list[str]:
    df = pd.read_csv(path_tsv, sep="\t", header=None)
    return df[0].astype(str).str.strip().str.replace(r"\.\d+$", "", regex=True).tolist()

def carregar_rotulos_filtrados(path_labels: Path) -> tuple[np.ndarray, np.ndarray]:
    with open(path_labels, "r", encoding="utf-8") as f:
        rotulos = np.array([int(float(l.strip().split(",")[0])) for l in f if l.strip()], dtype=np.int64)
    mascara = np.isin(rotulos, CLASSES_VALIDAS)
    y_filtrado = rotulos[mascara] - 1  # 1..7 -> 0..6
    return mascara, y_filtrado

def construir_modelo_mlp(input_dim: int = 600, n_classes: int = 7) -> Sequential:
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation="relu", name="camada_oculta_1"),
        Dense(64, activation="relu", name="camada_oculta_2"),
        Dense(32, activation="relu", name="camada_oculta_3"),
        Dense(n_classes, activation="softmax", name="camada_saida"),
    ], name="MLP_Classificador_Celular_600D")
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def plotar_curvas_aprendizado(history, titulo: str):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Acurácia ao Longo das Épocas", "Perda (Loss) ao Longo das Épocas")
    )
    epocas = list(range(1, len(history.history["accuracy"]) + 1))
    fig.add_trace(go.Scatter(x=epocas, y=history.history["accuracy"], mode="lines+markers", name="Treino Acurácia", line=dict(color="#1f77b4")), row=1, col=1)
    fig.add_trace(go.Scatter(x=epocas, y=history.history["val_accuracy"], mode="lines+markers", name="Validação Acurácia", line=dict(color="#ff7f0e")), row=1, col=1)
    fig.add_trace(go.Scatter(x=epocas, y=history.history["loss"], mode="lines+markers", name="Treino Loss", line=dict(color="#1f77b4")), row=1, col=2)
    fig.add_trace(go.Scatter(x=epocas, y=history.history["val_loss"], mode="lines+markers", name="Validação Loss", line=dict(color="#ff7f0e")), row=1, col=2)
    fig.update_layout(title_text=titulo, template="plotly_white", width=1000, height=450, legend=dict(orientation="h", y=-0.15))
    fig.update_xaxes(title_text="Época", dtick=1)
    fig.update_yaxes(title_text="Acurácia", row=1, col=1)
    fig.update_yaxes(title_text="Perda", row=1, col=2)
    return fig

def plotar_matriz_confusao(y_true: np.ndarray, y_pred: np.ndarray, titulo: str):
    cm_norm = confusion_matrix(y_true, y_pred, normalize="true")
    cm_abs = confusion_matrix(y_true, y_pred)
    text_labels = [[f"{abs_v}<br>({norm_v*100:.1f}%)" for abs_v, norm_v in zip(r_abs, r_norm)] for r_abs, r_norm in zip(cm_abs, cm_norm)]
    fig = go.Figure(data=go.Heatmap(
        z=cm_norm,
        x=NOMES_CLASSES,
        y=NOMES_CLASSES,
        colorscale="Blues",
        text=text_labels,
        texttemplate="%{text}",
        colorbar=dict(title="Recall"),
        zmin=0, zmax=1
    ))
    fig.update_layout(
        title=titulo,
        xaxis_title="Classe Predita",
        yaxis_title="Classe Real",
        yaxis_autorange="reversed",
        template="plotly_white",
        width=650, height=550
    )
    return fig

def plotar_comparativo(rel_limpo: dict, rel_ruido: dict, titulo: str, nome_cenario_ruido: str = "Com Ruído 0,5"):
    rotulos_plot = ["Acurácia Global", "Macro F1", "Weighted F1"] + [f"F1 {nome}" for nome in NOMES_CLASSES]
    
    vals_limpo = [
        rel_limpo["accuracy"],
        rel_limpo["macro avg"]["f1-score"],
        rel_limpo["weighted avg"]["f1-score"]
    ] + [rel_limpo.get(nome, {}).get("f1-score", 0.0) for nome in NOMES_CLASSES]
    
    vals_ruido = [
        rel_ruido["accuracy"],
        rel_ruido["macro avg"]["f1-score"],
        rel_ruido["weighted avg"]["f1-score"]
    ] + [rel_ruido.get(nome, {}).get("f1-score", 0.0) for nome in NOMES_CLASSES]
    
    fig = go.Figure(data=[
        go.Bar(name="Baseline (Sem Ruído)", x=rotulos_plot, y=vals_limpo, marker_color="#2ca02c"),
        go.Bar(name=nome_cenario_ruido, x=rotulos_plot, y=vals_ruido, marker_color="#d62728")
    ])
    fig.update_layout(
        title=titulo,
        barmode="group",
        yaxis=dict(title="Pontuação (0.0 a 1.0)", range=[0, 1.05]),
        template="plotly_white",
        width=1100, height=500,
        legend=dict(orientation="h", y=1.12)
    )
    return fig


In [3]:
# 1. Carregamento dos Dados, Rótulos e Ortobase da Referência
PATH_GENES_MATHYS = DADOS_DIR / "mathys" / "featuresM.tsv"
PATH_GENES_PAN = DADOS_DIR / "pan" / "featuresPANcorrigido.tsv"

genes_mathys = carregar_ensembl_ids(PATH_GENES_MATHYS)
genes_pan = carregar_ensembl_ids(PATH_GENES_PAN)

s_mathys = set(genes_mathys)
genes_comuns = [g for g in genes_pan if g in s_mathys]
genes_ausentes = [g for g in genes_pan if g not in s_mathys]
print(f"Referência (Pan)    : {len(genes_pan)} genes")
print(f"Alvo (Mathys)       : {len(genes_mathys)} genes")
print(f"Genes em Comum      : {len(genes_comuns)}")
print(f"Genes Ausentes (0.5): {len(genes_ausentes)} ({len(genes_ausentes)/len(genes_pan)*100:.2f}%)")

# Base Ortonormal OBRIGATORIAMENTE da REFERÊNCIA (Pan)
orthbase_ref = np.load(DADOS_DIR / "pan" / "orthbase_pan_600d.npy").astype(np.float32)
X_ref_limpo = np.load(DADOS_DIR / "pan" / "X_pan_600d.npy").astype(np.float32)
y_ref = np.load(DADOS_DIR / "pan" / "y_pan.npy")

# Projeção do Alvo (Mathys) no Espaço da Referência
X_alvo_ruido = np.load(DADOS_DIR / "mathys" / "X_mathys_alinhado_pan_ruido_600d.npy").astype(np.float32)
y_alvo = np.load(DADOS_DIR / "mathys" / "y_mathys.npy")

# TRAVAS DE SEGURANÇA (ASSERTIONS FAIL-FAST)
assert orthbase_ref.shape == (len(genes_pan), DIM_PROJECAO), f"Ortobase inválida: {orthbase_ref.shape}"
assert X_ref_limpo.shape == (len(y_ref), DIM_PROJECAO), f"X_ref inválido: {X_ref_limpo.shape}"
assert X_alvo_ruido.shape == (len(y_alvo), DIM_PROJECAO), f"X_alvo inválido: {X_alvo_ruido.shape}"
print("Travas de segurança e auditoria de ortobase passaram com 100% de sucesso!")


Referência (Pan)    : 61541 genes
Alvo (Mathys)       : 32738 genes
Genes em Comum      : 30836
Genes Ausentes (0.5): 30705 (49.89%)


Travas de segurança e auditoria de ortobase passaram com 100% de sucesso!


In [4]:
# 2. Particionamento e Escalonamento na Referência Limpa (Pan)
indices = np.arange(len(y_ref))
idx_treino_val, idx_teste = train_test_split(
    indices, test_size=0.15, random_state=SEED, stratify=y_ref
)
razao_val = 0.15 / (1.0 - 0.15)
idx_treino, idx_val = train_test_split(
    idx_treino_val, test_size=razao_val, random_state=SEED, stratify=y_ref[idx_treino_val]
)

scaler = StandardScaler()
X_treino_s = scaler.fit_transform(X_ref_limpo[idx_treino])
X_val_s = scaler.transform(X_ref_limpo[idx_val])
X_teste_limpo_s = scaler.transform(X_ref_limpo[idx_teste])

# Escalonamento dos dados do Alvo (Mathys) usando o Scaler da Referência
X_alvo_ruido_s = scaler.transform(X_alvo_ruido)

joblib.dump(scaler, DADOS_DIR / "pan" / "scaler_exp06.pkl")


['/home/rasert/workspace/Teste_MlP_fujita_para_fujitafaltante_fujitapos_imputacao/dados/pan/scaler_exp06.pkl']

In [5]:
# 3. Treinamento do Classificador MLP na Referência Limpa (Pan)
modelo = construir_modelo_mlp(input_dim=DIM_PROJECAO, n_classes=len(CLASSES_VALIDAS))
caminho_modelo = DADOS_DIR / "pan" / "modelo_mlp_exp06_pan_para_mathys.keras"
callbacks = [
    EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(str(caminho_modelo), monitor="val_loss", save_best_only=True, verbose=0)
]

t0 = time.time()
historico = modelo.fit(
    X_treino_s, y_ref[idx_treino],
    validation_data=(X_val_s, y_ref[idx_val]),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
print(f"Treinamento concluído em {time.time() - t0:.2f} segundos.")


Epoch 1/50


W0000 00:00:1792859250.096790 1090387 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


  1/629 ━━━━━━━━━━━━━━━━━━━━ 8:51 846ms/step - accuracy: 0.0156 - loss: 2.5872

 24/629 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6400 - loss: 1.1176    

 43/629 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7373 - loss: 0.7951

...

625/629 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9985 - loss: 0.0055

629/629 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9985 - loss: 0.0055 - val_accuracy: 0.9912 - val_loss: 0.0552


Epoch 6: early stopping


Restoring model weights from the end of the best epoch: 1.


Treinamento concluído em 10.96 segundos.


In [6]:
# 4. Gráfico 1 — Curvas de Aprendizado (Acurácia e Loss)
fig_aprendizado = plotar_curvas_aprendizado(historico, "Experimento 06: Curvas de Aprendizado MLP (Treinado no Pan Limpo)")
fig_aprendizado.show()


In [7]:
# 5. Avaliação Baseline Limpa (Conjunto de Teste Pan Holdout 15%)
probs_teste_limpo = modelo.predict(X_teste_limpo_s, verbose=0)
y_pred_teste_limpo = np.argmax(probs_teste_limpo, axis=1)

relatorio_limpo = classification_report(
    y_ref[idx_teste], y_pred_teste_limpo,
    target_names=NOMES_CLASSES,
    output_dict=True
)
print("=== RELATÓRIO BASELINE: PAN TESTE 15% ===")
print(classification_report(y_ref[idx_teste], y_pred_teste_limpo, target_names=NOMES_CLASSES))

fig_cm_limpo = plotar_matriz_confusao(
    y_ref[idx_teste], y_pred_teste_limpo,
    "Experimento 06: Matriz de Confusão — Pan Teste Limpo (Baseline)"
)
fig_cm_limpo.show()


=== RELATÓRIO BASELINE: PAN TESTE 15% ===
              precision    recall  f1-score   support

    Classe 1       1.00      1.00      1.00      2830
    Classe 2       0.90      0.90      0.90        82
    Classe 3       0.96      0.99      0.97       887
    Classe 4       0.96      0.92      0.94       403
    Classe 5       1.00      0.97      0.98       260
    Classe 6       1.00      1.00      1.00      3658
    Classe 7       0.98      0.98      0.98       499

    accuracy                           0.99      8619
   macro avg       0.97      0.96      0.97      8619
weighted avg       0.99      0.99      0.99      8619



In [8]:
# 6. Avaliação Cross-Dataset com Ruído (100% Células Válidas do Mathys)
probs_alvo_ruido = modelo.predict(X_alvo_ruido_s, verbose=0)
y_pred_alvo_ruido = np.argmax(probs_alvo_ruido, axis=1)

relatorio_ruido = classification_report(
    y_alvo, y_pred_alvo_ruido,
    target_names=NOMES_CLASSES,
    output_dict=True
)
print("=== RELATÓRIO: MATHYS COMPLETO COM RUÍDO 0,5 (INFERÊNCIA PAN -> MATHYS) ===")
print(classification_report(y_alvo, y_pred_alvo_ruido, target_names=NOMES_CLASSES))

fig_cm_ruido = plotar_matriz_confusao(
    y_alvo, y_pred_alvo_ruido,
    "Experimento 06: Matriz de Confusão — Mathys Completo com Ruído 0,5"
)
fig_cm_ruido.show()


=== RELATÓRIO: MATHYS COMPLETO COM RUÍDO 0,5 (INFERÊNCIA PAN -> MATHYS) ===
              precision    recall  f1-score   support

    Classe 1       0.28      1.00      0.44      2597
    Classe 2       0.05      0.37      0.08        68
    Classe 3       0.99      0.86      0.92     24548
    Classe 4       0.89      0.55      0.68      6420
    Classe 5       1.00      0.01      0.03       485
    Classe 6       0.93      0.99      0.96      9218
    Classe 7       0.97      0.38      0.55      2236

    accuracy                           0.82     45572
   macro avg       0.73      0.59      0.52     45572
weighted avg       0.92      0.82      0.84     45572



In [9]:
# 7. Gráfico 4 — Comparativo de Desempenho (Pan Limpo vs. Mathys com Ruído 0,5)
fig_comp = plotar_comparativo(
    relatorio_limpo, relatorio_ruido,
    "Experimento 06 (Pan -> Mathys): Queda Cross-Dataset com Ruído 0,5",
    nome_cenario_ruido="Mathys com Ruído 0,5"
)
fig_comp.show()

acc_limpo = relatorio_limpo["accuracy"]
acc_ruido = relatorio_ruido["accuracy"]
macro_limpo = relatorio_limpo["macro avg"]["f1-score"]
macro_ruido = relatorio_ruido["macro avg"]["f1-score"]

print("=" * 60)
print(f"Acurácia Global   : {acc_limpo*100:.2f}% (Pan Limpo) -> {acc_ruido*100:.2f}% (Mathys Ruído) | Delta: {(acc_limpo - acc_ruido)*100:+.2f} p.p.")
print(f"Macro F1-Score    : {macro_limpo*100:.2f}% -> {macro_ruido*100:.2f}% | Delta: {(macro_limpo - macro_ruido)*100:+.2f} p.p.")
print("=" * 60)


Acurácia Global   : 98.94% (Pan Limpo) -> 81.52% (Mathys Ruído) | Delta: +17.42 p.p.
Macro F1-Score    : 96.74% -> 52.21% | Delta: +44.53 p.p.
